#  Download NetCDF and conversion to Parquet 

- Data source: https://data-argo.ifremer.fr/geo/

- Based on: https://euroargodev.github.io/argoonlineschool/Lessons/L03_UsingArgoData/Chapter33_ArgoDatabyDate_Intro.html

Downloads raw Argo profile NetCDF files from the Ifremer GDAC, parses them into a
single tidy Parquet file (one row per observation level), and cleans up QC-flag
encoding issues along the way (byte-strings like `b'4'` -> `4`, `JULD_QC`/`POSITION_QC`
stored as strings instead of ints).

**This notebook is parameterized by ocean basin and year range** (see the Config
cell below. Change `OCEAN_NAME` / `YEAR_START` / `YEAR_END` and every path,
filename, and the notebook updates automatically.)

Pipeline stages:
1. Download raw `.nc` files from Ifremer (parallel)
2. Parse NetCDF -> per-batch Parquet files (low RAM, incremental)
3. Merge batch Parquet files into one file
4. Clean QC byte-string encoding (`b'4'` -> `4`)
5. Fix `JULD_QC` / `POSITION_QC` dtype (string -> nullable Int8)

## 0 · Config, imports & derived paths

In [1]:
import gc
import os
import re
import shutil
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xarray as xr
from tqdm.notebook import tqdm

In [ ]:
# CONFIG: edit these to change the ocean and year range to process
OCEAN_NAME = "atlantic"      # atlantic, pacific, indian
YEAR_START = 2021 #2018
YEAR_END   = 2025 #2022

# ---- Directory layout -------------------------------------------------------
#   /work/drgarcia/Dataset/<ocean>_ocean/<year_start>-<year_end>/raw
#   /work/drgarcia/Dataset/<ocean>_ocean/<year_start>-<year_end>/processed
DATASET_ROOT       = Path("/work/drgarcia/Dataset")
BASE_DIR           = DATASET_ROOT / f"{OCEAN_NAME}_ocean" / f"{YEAR_START}-{YEAR_END}"
RAW_DATA_DIR        = BASE_DIR / "raw"
PROCESSED_DATA_DIR  = BASE_DIR / "1.processed"
TEMP_DIR             = PROCESSED_DATA_DIR / "temp_batches"   # per-batch parquet files, deleted after merge

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

YEARS  = list(range(YEAR_START, YEAR_END + 1))
MONTHS = list(range(1, 13))

BATCH_SIZE  = 20   # NetCDF files per processing batch (Stage 2)
MAX_WORKERS = 10   # parallel download / IO threads

# -Output filenames, files named per stage 
# These are derived automatically from OCEAN_NAME/YEAR_START/YEAR_END
MERGED_PARQUET    = PROCESSED_DATA_DIR / f"{OCEAN_NAME}_{YEAR_START}_{YEAR_END}_merged.parquet"
QC_CLEAN_PARQUET  = PROCESSED_DATA_DIR / f"{OCEAN_NAME}_{YEAR_START}_{YEAR_END}_qc_clean.parquet"
QC_CLEAN2_PARQUET = PROCESSED_DATA_DIR / f"{OCEAN_NAME}_{YEAR_START}_{YEAR_END}_clean2.parquet"

# -Argo variable groups, by how they're shaped inside the NetCDF file, for more info review argo user manual
# FILE_GLOBAL_VARS : one value for the whole file
# PROFILE_VARS     : one value per profile (broadcast across all levels of that profile)
# LEVEL_VARS       : one value per (profile, level) pair — the actual measurements
FILE_GLOBAL_VARS = ['DATA_TYPE', 'REFERENCE_DATE_TIME', 'DATE_CREATION', 'DATE_UPDATE']

PROFILE_VARS = [
    'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'DATA_CENTRE', 'DATA_MODE',
    'PLATFORM_TYPE', 'JULD', 'JULD_QC', 'JULD_LOCATION',
    'LATITUDE', 'LONGITUDE', 'POSITION_QC',
    'PROFILE_PRES_QC', 'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC',
]

LEVEL_VARS = [
    'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR',
    'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'TEMP_ADJUSTED_ERROR',
    'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC', 'PSAL_ADJUSTED_ERROR',
]

VARS_INTERES = FILE_GLOBAL_VARS + PROFILE_VARS + LEVEL_VARS  # kept for diagnostics/reference

# QC columns cleaned at Stage 4 (byte-string -> Int8)
QC_COLUMNS = ['PRES_QC', 'TEMP_QC', 'PSAL_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC']

# Columns fixed at Stage 5 (stored as plain strings instead of QC ints)
STR_TO_INT8_COLUMNS = ["JULD_QC", "POSITION_QC"]

print(f"Ocean       : {OCEAN_NAME}")
print(f"Years       : {YEAR_START}-{YEAR_END}")
print(f"Raw dir     : {RAW_DATA_DIR}")
print(f"Processed   : {PROCESSED_DATA_DIR}")

Ocean       : atlantic
Years       : 2021-2025
Raw dir     : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/raw
Processed   : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed


## 1 · Download from Ifremer (parallel)

In [3]:
def _download_single_file(args):
    """Download one file; skip if it already exists on disk."""
    file_url, target_path = args
    if target_path.exists():
        return "skip"
    try:
        urllib.request.urlretrieve(file_url, target_path)
        return "ok"
    except Exception:
        return "fail"


def download_argo_month(year: int, month: int, base_dir: Path, max_workers: int = 10) -> int:
    """Download every NetCDF profile file for one (year, month) from the Ifremer GDAC.

    Returns the number of newly-downloaded files (already-present files are skipped).
    """
    month_str  = f"{month:02d}"
    url        = f"https://data-argo.ifremer.fr/geo/{OCEAN_NAME}_ocean/{year}/{month_str}/"
    output_dir = base_dir / str(year) / month_str
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            html = resp.read().decode("utf-8")

        files = re.findall(r'href="([^"]+\.nc)"', html)
        if not files:
            print(f"  {year}/{month_str}  (error) no .nc files found")
            return 0

        tasks = [(url + f, output_dir / f) for f in files]

        results = {"ok": 0, "skip": 0, "fail": 0}
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for r in ex.map(_download_single_file, tasks):
                results[r] += 1

        print(f"  {year}/{month_str}  total={len(files)}  "
              f"new={results['ok']}  skipped={results['skip']}  failed={results['fail']}")
        return results["ok"]

    except Exception as e:
        print(f"  {year}/{month_str}  x {str(e)[:60]}")
        return 0

In [4]:
print("Starting download...\n")
total_new = 0
for year in YEARS:
    print(f"-- {year} --")
    for month in MONTHS:
        total_new += download_argo_month(year, month, RAW_DATA_DIR, MAX_WORKERS)

nc_files = list(RAW_DATA_DIR.rglob("*.nc"))
print(f"\nDownload complete. New files: {total_new}")
print(f"Total .nc files on disk: {len(nc_files):,}")

Starting download...

-- 2021 --
  2021/01  total=31  new=0  skipped=31  failed=0
  2021/02  total=28  new=0  skipped=28  failed=0
  2021/03  total=31  new=0  skipped=31  failed=0
  2021/04  total=30  new=0  skipped=30  failed=0
  2021/05  total=31  new=0  skipped=31  failed=0


  2021/06  total=30  new=0  skipped=30  failed=0
  2021/07  total=31  new=0  skipped=31  failed=0


  2021/08  total=31  new=0  skipped=31  failed=0
  2021/09  total=30  new=0  skipped=30  failed=0
  2021/10  total=31  new=0  skipped=31  failed=0
  2021/11  total=30  new=0  skipped=30  failed=0
  2021/12  total=31  new=0  skipped=31  failed=0
-- 2022 --
  2022/01  total=31  new=0  skipped=31  failed=0


  2022/02  total=28  new=0  skipped=28  failed=0
  2022/03  total=31  new=0  skipped=31  failed=0
  2022/04  total=30  new=0  skipped=30  failed=0


  2022/05  total=31  new=0  skipped=31  failed=0
  2022/06  total=30  new=0  skipped=30  failed=0
  2022/07  total=31  new=0  skipped=31  failed=0
  2022/08  total=31  new=0  skipped=31  failed=0
  2022/09  total=30  new=0  skipped=30  failed=0
  2022/10  total=31  new=0  skipped=31  failed=0


...

  2024/06  total=30  new=0  skipped=30  failed=0
  2024/07  total=31  new=0  skipped=31  failed=0
  2024/08  total=31  new=0  skipped=31  failed=0
  2024/09  total=30  new=0  skipped=30  failed=0
  2024/10  total=31  new=0  skipped=31  failed=0
  2024/11  total=30  new=0  skipped=30  failed=0
  2024/12  total=31  new=0  skipped=31  failed=0
-- 2025 --


  2025/01  total=31  new=0  skipped=31  failed=0


  2025/02  total=28  new=0  skipped=28  failed=0
  2025/03  total=31  new=0  skipped=31  failed=0
  2025/04  total=30  new=0  skipped=30  failed=0
  2025/05  total=31  new=0  skipped=31  failed=0
  2025/06  total=30  new=0  skipped=30  failed=0
  2025/07  total=31  new=0  skipped=31  failed=0
  2025/08  total=31  new=0  skipped=31  failed=0
  2025/09  total=30  new=0  skipped=30  failed=0


  2025/10  total=31  new=0  skipped=31  failed=0


  2025/11  total=30  new=0  skipped=30  failed=0
  2025/12  total=31  new=0  skipped=31  failed=0

Download complete. New files: 0
Total .nc files on disk: 1,826


## 2 · Read NetCDF -> Parquet (incremental, low RAM)

`read_argo_netcdf` flattens one profile file into a tidy DataFrame with one row per
(profile, level) observation. It tries three xarray engines in order (`netcdf4`,
`h5netcdf`, `scipy`) since some Argo files are more permissive with one than another.
Files are processed in batches of `BATCH_SIZE` and written straight to disk as
per-batch Parquet files under `TEMP_DIR`, so memory use stays flat regardless of how
many files there are in total.

In [5]:
def _element_to_str(c) -> str:
    """Decode a single NetCDF character/byte element to a clean Python string."""
    if isinstance(c, (bytes, np.bytes_)):
        return c.decode("utf-8", errors="replace").replace("\x00", "")
    if isinstance(c, np.ndarray):
        return "".join(_element_to_str(x) for x in c.flat)
    return str(c).replace("\x00", "")


def _decode_chararray(arr: np.ndarray) -> np.ndarray:
    """Decode a NetCDF fixed-width char array (1D or 2D) into an object array of strings."""
    if arr.ndim == 2:
        return np.array(
            ["".join(_element_to_str(c) for c in row).strip() for row in arr],
            dtype=object,
        )
    if arr.ndim == 1:
        out = []
        for elem in arr:
            if isinstance(elem, np.ndarray):
                s = "".join(_element_to_str(c) for c in elem.flat)
            elif isinstance(elem, (bytes, np.bytes_)):
                s = elem.decode("utf-8", errors="replace").replace("\x00", "")
            else:
                s = str(elem).replace("\x00", "")
            out.append(s.strip())
        return np.array(out, dtype=object)
    return arr


def _decode_global_string(arr: np.ndarray) -> str:
    """Decode a file-level (0-3 dim) char array into a single string."""
    if arr is None:
        return ""
    if arr.ndim == 0:
        return _element_to_str(arr.item()).strip()
    if arr.ndim == 1:
        return "".join(_element_to_str(c) for c in arr).strip()
    if arr.ndim == 2 and arr.shape[0] == 1:
        return "".join(_element_to_str(c) for c in arr[0]).strip()
    return "".join(_element_to_str(c) for c in arr.flat).strip()


def read_argo_netcdf(file_path: Path) -> "pd.DataFrame | None":
    """Parse one Argo profile NetCDF file into a tidy (profile x level) DataFrame.

    Returns None if the file has no usable profile/level data or fails to open
    with every available engine (the failure is printed with the file name).
    """
    engines = ["netcdf4", "h5netcdf", "scipy"]
    last_error = None

    for engine in engines:
        try:
            with xr.open_dataset(file_path, engine=engine, mask_and_scale=True) as ds:
                dims     = dict(ds.sizes)
                n_prof   = dims.get("N_PROF", 0)
                n_levels = dims.get("N_LEVELS", 0)

                if n_prof == 0 or n_levels == 0:
                    return None

                row_dict   = {}
                total_rows = n_prof * n_levels

                # File-global vars: same value repeated for every row in the file
                for var in FILE_GLOBAL_VARS:
                    if var not in ds:
                        row_dict[var] = np.full(total_rows, np.nan, dtype=object)
                        continue
                    val_str = _decode_global_string(ds[var].values)
                    row_dict[var] = np.full(total_rows, val_str if val_str else np.nan, dtype=object)

                # Profile-level vars: one value per profile, broadcast across its levels
                for var in PROFILE_VARS:
                    if var not in ds:
                        continue
                    arr = ds[var].values
                    if arr.dtype.kind in ("S", "U", "O", "b") or (arr.ndim == 1 and arr.dtype.kind == "O"):
                        arr = _decode_chararray(arr)
                    if arr.shape[0] != n_prof or arr.ndim > 1:
                        continue
                    row_dict[var] = np.repeat(arr, n_levels)

                # Level vars: the actual (profile, level) measurements
                for var in LEVEL_VARS:
                    if var not in ds:
                        continue
                    arr = ds[var].values
                    if arr.shape != (n_prof, n_levels):
                        continue
                    row_dict[var] = arr.ravel()

                if not row_dict:
                    return None

                df = pd.DataFrame(row_dict)

                if "PLATFORM_NUMBER" in df.columns:
                    df["PLATFORM_NUMBER"] = (
                        df["PLATFORM_NUMBER"].astype(str).str.strip().str.replace(r"\x00", "", regex=True)
                    )

                return df if not df.empty else None

        except Exception as e:
            last_error = e
            continue  # try next engine

    print(f"  [SKIP] {file_path.name}: {type(last_error).__name__}: {str(last_error)[:150]}")
    return None

In [6]:
# ---- Sanity check on a few files before committing to the full run ----
nc_files = sorted(RAW_DATA_DIR.rglob("*.nc"))
print(f"NetCDF files found: {len(nc_files):,}")

print("\n-- Sanity check on first 3 files --")
for f in nc_files[:3]:
    df = read_argo_netcdf(f)
    if df is None:
        print(f"  {f.name}  ->  None")
    else:
        print(f"  {f.name}  ->  {df.shape}")
        print(f"    PRES range   : {df['PRES'].min():.1f} - {df['PRES'].max():.1f}")
        print(f"    Platforms    : {df['PLATFORM_NUMBER'].nunique()}")
        print(f"    NaN PRES %   : {df['PRES'].isna().mean() * 100:.1f}%")
        print(f"    DATA_TYPE    : {df['DATA_TYPE'].iloc[0]}")
        print(f"    DATE_CREATION: {df['DATE_CREATION'].iloc[0]}")
        print(f"    Columns      : {list(df.columns)}")

NetCDF files found: 1,826

-- Sanity check on first 3 files --


  20210101_prof.nc  ->  (352352, 34)
    PRES range   : -0.3 - 5606.6
    Platforms    : 168
    NaN PRES %   : 73.2%
    DATA_TYPE    : Argo profile
    DATE_CREATION: 20210101042707
    Columns      : ['DATA_TYPE', 'REFERENCE_DATE_TIME', 'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'DATA_CENTRE', 'DATA_MODE', 'PLATFORM_TYPE', 'JULD', 'JULD_QC', 'JULD_LOCATION', 'LATITUDE', 'LONGITUDE', 'POSITION_QC', 'PROFILE_PRES_QC', 'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC', 'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR', 'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'TEMP_ADJUSTED_ERROR', 'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC', 'PSAL_ADJUSTED_ERROR']
  20210102_prof.nc  ->  (270825, 34)


    PRES range   : -0.3 - 5454.9
    Platforms    : 153
    NaN PRES %   : 71.4%
    DATA_TYPE    : Argo profile
    DATE_CREATION: 20210102042709
    Columns      : ['DATA_TYPE', 'REFERENCE_DATE_TIME', 'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'DATA_CENTRE', 'DATA_MODE', 'PLATFORM_TYPE', 'JULD', 'JULD_QC', 'JULD_LOCATION', 'LATITUDE', 'LONGITUDE', 'POSITION_QC', 'PROFILE_PRES_QC', 'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC', 'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR', 'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'TEMP_ADJUSTED_ERROR', 'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC', 'PSAL_ADJUSTED_ERROR']
  20210103_prof.nc  ->  (250000, 34)
    PRES range   : -0.3 - 5612.5
    Platforms    : 122
    NaN PRES %   : 71.4%
    DATA_TYPE    : Argo profile
    DATE_CREATION: 20210103022704
    Columns      : ['DATA_TYPE', 'REFERENCE_DATE_TIME', 'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'CYCLE_NUMB

In [7]:
# ---- Full batch processing: NetCDF -> per-batch Parquet under TEMP_DIR ----
print(f"\n{'=' * 60}")
print("Starting full batch processing...")
print(f"{'=' * 60}")

TEMP_DIR.mkdir(parents=True, exist_ok=True)

empty_batches = []
for i, start in enumerate(tqdm(range(0, len(nc_files), BATCH_SIZE), desc="Processing batches")):
    batch_files = nc_files[start: start + BATCH_SIZE]
    batch_dfs   = []

    for nc_file in batch_files:
        df_temp = read_argo_netcdf(nc_file)
        if df_temp is not None:
            batch_dfs.append(df_temp)

    if not batch_dfs:
        empty_batches.append(i)
        continue

    batch_df = pd.concat(batch_dfs, ignore_index=True)
    batch_df.to_parquet(TEMP_DIR / f"batch_{i:05d}.parquet", index=False, compression="snappy")
    del batch_dfs, batch_df
    gc.collect()

written = len(list(TEMP_DIR.glob("*.parquet")))
print(f"\nBatches written      : {written}")
print(f"Empty batches skipped: {len(empty_batches)}")
if empty_batches:
    print(f"  Batch indices      : {empty_batches[:20]}")


Starting full batch processing...


Processing batches:   0%|          | 0/92 [00:00<?, ?it/s]


Batches written      : 92
Empty batches skipped: 0


### Optional diagnostics (run only if the batch processing above returns `None`/empty for every file)

Useful when a new ocean/year range has different variable names, all-NaN
measurement columns, or files that fail to open with every xarray engine.

In [8]:
def diagnose_netcdf_variables(raw_dir: Path, vars_of_interest: list[str], n_inspect: int = 5, n_sample: int = 50):
    """Inspect a handful of raw NetCDF files to explain why read_argo_netcdf might
    be returning None for every file: missing variables, name mismatches, or
    all-NaN measurement columns.
    """
    nc_files = sorted(raw_dir.rglob("*.nc"))
    print(f"Total .nc files found: {len(nc_files)}")
    if not nc_files:
        print("ERROR: No .nc files found. Check raw_dir path.")
        return

    print(f"\n{'=' * 70}\nInspecting first {n_inspect} files\n{'=' * 70}")
    for nc_file in nc_files[:n_inspect]:
        print(f"\n-- {nc_file.name}")
        print(f"   Path : {nc_file}")
        print(f"   Size : {nc_file.stat().st_size / 1024:.1f} KB")

        try:
            with xr.open_dataset(nc_file, engine="netcdf4") as ds:
                print(f"   Dims : {dict(ds.sizes)}")

                all_vars = list(ds.data_vars)
                print(f"   Variables ({len(all_vars)}): {all_vars[:30]}")

                found   = [v for v in vars_of_interest if v in ds]
                missing = [v for v in vars_of_interest if v not in ds]
                print(f"   vars_of_interest found   ({len(found)}): {found}")
                print(f"   vars_of_interest missing ({len(missing)}): {missing[:20]}")

                if not found:
                    print("   *** NONE of vars_of_interest found — this is why read_argo_netcdf returns None ***")
                    print("   Likely cause: variable names differ, or this is a different Argo profile type")
                    continue

                for var in ["PRES", "TEMP", "PSAL"]:
                    if var in ds:
                        arr = ds[var].values
                        nan_pct = np.isnan(arr.astype(float)).mean() * 100
                        print(f"   {var}: shape={arr.shape}  dtype={arr.dtype}  NaN%={nan_pct:.1f}")
                    else:
                        print(f"   {var}: NOT PRESENT")

                try:
                    df_test = ds[found].to_dataframe().reset_index()
                    print(f"   to_dataframe shape: {df_test.shape}")
                    meas_cols = [c for c in ["PRES", "TEMP", "PSAL"] if c in df_test.columns]
                    if meas_cols:
                        before = len(df_test)
                        df_test = df_test.dropna(subset=meas_cols, how="all")
                        print(f"   Rows after dropna(PRES/TEMP/PSAL all-NaN): {len(df_test)} (was {before})")
                    else:
                        print("   *** No PRES/TEMP/PSAL in DataFrame columns after to_dataframe ***")
                except Exception as e:
                    print(f"   to_dataframe ERROR: {e}")

        except Exception as e:
            print(f"   OPEN ERROR: {e}")

    print(f"\n{'=' * 70}\nVariable presence count across first {n_sample} files\n{'=' * 70}")
    var_count = Counter()
    for nc_file in nc_files[:n_sample]:
        try:
            with xr.open_dataset(nc_file, engine="netcdf4") as ds:
                for v in ds.data_vars:
                    var_count[v] += 1
        except Exception:
            pass

    for var, cnt in sorted(var_count.items(), key=lambda x: -x[1]):
        marker = " (expected)" if var in vars_of_interest else ""
        print(f"  {cnt:>3}/{n_sample}  {var}{marker}")

    print(f"\n{'=' * 70}\nChecking for name mismatches on key variables\n{'=' * 70}")
    all_found_vars = set(var_count.keys())
    for v in ["PRES", "TEMP", "PSAL", "PLATFORM_NUMBER", "CYCLE_NUMBER", "JULD"]:
        if v not in all_found_vars:
            candidates = [x for x in all_found_vars if x.upper() == v.upper() or v.lower() in x.lower()]
            print(f"  {v} NOT FOUND — possible matches: {candidates}")
        else:
            print(f"  {v} found OK")


def check_nc_file_readable(file_path: Path):
    """Quick sanity check: can netCDF4 even open this file, and what variables does it see?"""
    import netCDF4
    try:
        ds = netCDF4.Dataset(file_path)
        print(f"{file_path.name}: OK, vars={list(ds.variables.keys())[:5]}")
        ds.close()
    except Exception as e:
        print(f"{file_path.name}: {type(e).__name__}: {e}")
        print(f"  Size: {file_path.stat().st_size} bytes")


def inspect_history_vars(raw_dir: Path, file_index: int = 500):
    """Peek at the HISTORY_* variables of one raw file (usually dropped from the
    final dataset, but useful to confirm what they look like before deciding to
    drop them).
    """
    nc_files = sorted(raw_dir.rglob("*.nc"))
    if not nc_files:
        print("No .nc files found.")
        return
    f = nc_files[min(file_index, len(nc_files) - 1)]
    with xr.open_dataset(f, engine="netcdf4") as ds:
        print("File:", f.name)
        print("N_HISTORY:", ds.sizes.get("N_HISTORY", "not present"))
        print("HISTORY vars:", [v for v in ds.data_vars if "HISTORY" in v])
        if "HISTORY_INSTITUTION" in ds:
            arr = ds["HISTORY_INSTITUTION"].values
            print("shape:", arr.shape)
            print("dtype:", arr.dtype)
            print("sample:", arr[:2])


# Uncomment to run:
# diagnose_netcdf_variables(RAW_DATA_DIR, VARS_INTERES)
# check_nc_file_readable(nc_files[0])
# inspect_history_vars(RAW_DATA_DIR)

## 3 · Merge batch parquets -> single file

In [9]:
parquet_files = sorted(TEMP_DIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} batch files ready to merge")

writer = None
for pf in tqdm(parquet_files, desc="Merging"):
    table = pq.read_table(pf)
    if writer is None:
        writer = pq.ParquetWriter(MERGED_PARQUET, table.schema, compression="snappy")
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()

meta = pq.read_metadata(MERGED_PARQUET)
print(f"\nReady. Merged Parquet: {meta.num_rows:,} rows")
print(f"Size: {MERGED_PARQUET.stat().st_size / 1e9:.2f} GB")

Found 92 batch files ready to merge


Merging:   0%|          | 0/92 [00:00<?, ?it/s]


Ready. Merged Parquet: 689,217,838 rows
Size: 2.37 GB


In [10]:
# Cleanup temp batch files — kept as its own cell (destructive) so you only run
# it once you've confirmed the merge above succeeded.
shutil.rmtree(TEMP_DIR)
print("Temp batch files removed.")

Temp batch files removed.


## 4 · Parquet inspection utility

In [11]:
def inspect_parquet(path: Path, deep: bool = False, max_unique_track: int = 500):
    """Print a summary of a Parquet file: size, row/column counts, per-column
    dtype + sample values, and (with deep=True) exact unique-value counts and
    null counts scanned across every row group instead of just the first.

    deep=False (default): fast, reads only row group 0 — good for a quick look.
    deep=True: scans the whole file chunk by chunk — good for a final QA pass.
    """
    path = Path(path)
    pfile = pq.ParquetFile(path)
    meta  = pfile.metadata

    print("-" * 60)
    print("PARQUET SUMMARY")
    print("-" * 60)
    print(f"File       : {path}")
    print(f"Size       : {path.stat().st_size / 1024 ** 3:.3f} GB")
    print(f"Total rows : {meta.num_rows:,}")
    print(f"Row groups : {meta.num_row_groups}")
    print(f"Columns    : {meta.num_columns}")

    col_names = [meta.row_group(0).column(i).path_in_schema for i in range(meta.num_columns)]
    df_sample = pfile.read_row_group(0).to_pandas()

    if not deep:
        rows = []
        for i, col in enumerate(col_names):
            values = df_sample[col].unique() if col in df_sample.columns else np.array([])
            sample = list(values[:5])
            if len(values) > 5:
                sample.append("...")
            rows.append({
                "Column": col,
                "Type Parquet": str(pfile.schema.column(i).physical_type),
                "Type Pandas": str(df_sample[col].dtype) if col in df_sample.columns else "?",
                "Unique Values (row group 0)": len(values),
                "Sample values": sample,
            })
        with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.max_colwidth", None):
            display(pd.DataFrame(rows))

        print("\nFirst 5 rows (row group 0 only):")
        with pd.option_context("display.max_columns", None):
            display(df_sample.head())
        return

    # ---- deep scan across all row groups ----
    print("\nScanning unique values + nulls across all row groups...")
    unique_sets = {col: set() for col in col_names}
    overflow    = {col: False for col in col_names}
    null_counts = {col: 0 for col in col_names}
    total_read  = 0

    for rg in range(meta.num_row_groups):
        batch = pfile.read_row_group(rg).to_pandas()
        total_read += len(batch)
        for col in col_names:
            if col not in batch.columns:
                continue
            s = batch[col]
            null_counts[col] += int(s.isna().sum())
            if not overflow[col]:
                unique_sets[col] |= set(s.dropna().unique())
                if len(unique_sets[col]) > max_unique_track:
                    overflow[col] = True
                    unique_sets[col] = set()
        del batch

    print(f"  Rows read: {total_read:,}")

    rows = []
    for i, col in enumerate(col_names):
        dtype_pd = str(df_sample[col].dtype) if col in df_sample.columns else "?"
        dtype_pq = str(pfile.schema.column(i).physical_type)
        n_nulls  = null_counts[col]
        pct_null = 100 * n_nulls / meta.num_rows if meta.num_rows else 0

        if overflow[col]:
            n_unique, sample = f"> {max_unique_track}", "(too many to list)"
        else:
            vals = sorted(unique_sets[col], key=str)
            n_unique = len(vals)
            sample = vals[:5] + (["..."] if len(vals) > 5 else [])

        rows.append({
            "Column": col,
            "Type Parquet": dtype_pq,
            "Type Pandas": dtype_pd,
            "Unique Values": n_unique,
            "Sample values": sample,
            "Nulls": n_nulls,
            "Nulls %": round(pct_null, 2),
        })

    with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows))

## 5 · Verify merged output

In [12]:
inspect_parquet(MERGED_PARQUET, deep=True)

------------------------------------------------------------
PARQUET SUMMARY
------------------------------------------------------------
File       : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_merged.parquet
Size       : 2.203 GB
Total rows : 689,217,838
Row groups : 704
Columns    : 34



Scanning unique values + nulls across all row groups...


  Rows read: 689,217,838


,Column,Type Parquet,Type Pandas,Unique Values,Sample values,Nulls,Nulls %
0,DATA_TYPE,BYTE_ARRAY,object,1,[Argo profile],0,0.00
1,REFERENCE_DATE_TIME,BYTE_ARRAY,object,1,[19500101000000],0,0.00
2,DATE_CREATION,BYTE_ARRAY,object,> 500,(too many to list),0,0.00
3,DATE_UPDATE,BYTE_ARRAY,object,> 500,(too many to list),0,0.00
4,PLATFORM_NUMBER,BYTE_ARRAY,object,> 500,(too many to list),0,0.00
5,CYCLE_NUMBER,DOUBLE,float64,> 500,(too many to list),0,0.00
6,DIRECTION,BYTE_ARRAY,object,2,"[A, D]",0,0.00
7,DATA_CENTRE,BYTE_ARRAY,object,8,"[AO, BO, CS, HZ, IF, ...]",0,0.00
8,DATA_MODE,BYTE_ARRAY,object,3,"[A, D, R]",0,0.00
9,PLATFORM_TYPE,BYTE_ARRAY,object,20,"[ALTO, APEX, APEX_D, ARVOR, ARVOR_C, ...]",0,0.00


## 6 · Clean QC byte-string encoding (`b'4'` -> `4`)

QC flag columns sometimes come out of `to_dataframe()` as Python byte-string
reprs (`"b'4'"`) instead of plain integers. `_clean_qc_col` strips that wrapper
and casts to nullable `Int8`.

In [13]:
def _clean_qc_col(series: pd.Series) -> pd.Series:
    """Normalize a QC column to nullable Int8, handling values like "b'4'", "4", NaN."""
    if series.dtype == object:
        return (
            series
            .astype(str)
            .str.strip()
            .str.replace(r"^b'(.+)'$", r"\1", regex=True)
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
            .astype("Int8")
        )
    return series.astype("Int8")


pfile = pq.ParquetFile(MERGED_PARQUET)
n_rg  = pfile.metadata.num_row_groups
cols_to_keep = pfile.schema.names

print(f"Input      : {MERGED_PARQUET}")
print(f"Output     : {QC_CLEAN_PARQUET}")
print(f"Row groups : {n_rg}  |  Rows: {pfile.metadata.num_rows:,}")
print(f"QC columns to clean: {QC_COLUMNS}\n")

writer = None
for rg in range(n_rg):
    df = pfile.read_row_group(rg, columns=cols_to_keep).to_pandas()

    for col in QC_COLUMNS:
        if col not in df.columns:
            print(f"  [SKIP] '{col}' not found in row group {rg}")
            continue
        before = df[col].dtype
        df[col] = _clean_qc_col(df[col])
        if rg == 0:
            sample_before = pfile.read_row_group(0, columns=[col]).to_pandas()[col].head(3).tolist()
            sample_after  = df[col].head(3).tolist()
            print(f"  {col}: {before} -> {df[col].dtype}  |  {sample_before} -> {sample_after}")

    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(QC_CLEAN_PARQUET, table.schema, compression="snappy")
    writer.write_table(table)
    print(f"  Row group {rg + 1}/{n_rg} written", end="\r")

if writer:
    writer.close()

print(f"\n\nDone: {QC_CLEAN_PARQUET}")

Input      : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_merged.parquet
Output     : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_qc_clean.parquet
Row groups : 704  |  Rows: 689,217,838
QC columns to clean: ['PRES_QC', 'TEMP_QC', 'PSAL_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC']



  PRES_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  TEMP_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  PSAL_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  PRES_ADJUSTED_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  TEMP_ADJUSTED_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  PSAL_ADJUSTED_QC: object -> Int8  |  [b'1', b'1', b'1'] -> [1, 1, 1]


  Row group 70/704 written


  Row group 429/704 written


  Row group 704/704 written

Done: /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_qc_clean.parquet


## 7 · Verify QC-cleaned output

In [14]:
inspect_parquet(QC_CLEAN_PARQUET, deep=False)

------------------------------------------------------------
PARQUET SUMMARY
------------------------------------------------------------
File       : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_qc_clean.parquet
Size       : 2.207 GB
Total rows : 689,217,838
Row groups : 704
Columns    : 34


,Column,Type Parquet,Type Pandas,Unique Values (row group 0),Sample values
0,DATA_TYPE,BYTE_ARRAY,object,1,[Argo profile]
1,REFERENCE_DATE_TIME,BYTE_ARRAY,object,1,[19500101000000]
2,DATE_CREATION,BYTE_ARRAY,object,4,"[20210101042707, 20210102042709, 20210103022704, 20210104052705]"
3,DATE_UPDATE,BYTE_ARRAY,object,4,"[20260602012447, 20260604064250, 20260527033716, 20260603030153]"
4,PLATFORM_NUMBER,BYTE_ARRAY,object,483,"[5905982, 5905980, 3901219, 6902846, 3901105, ...]"
5,CYCLE_NUMBER,DOUBLE,float64,227,"[81.0, 140.0, 201.0, 200.0, 69.0, ...]"
6,DIRECTION,BYTE_ARRAY,object,2,"[A, D]"
7,DATA_CENTRE,BYTE_ARRAY,object,7,"[AO, IF, BO, JA, ME, ...]"
8,DATA_MODE,BYTE_ARRAY,object,3,"[D, A, R]"
9,PLATFORM_TYPE,BYTE_ARRAY,object,15,"[APEX, S2A, ARVOR, PROVOR_III, SOLO_D_MRV, ...]"



First 5 rows (row group 0 only):


,DATA_TYPE,REFERENCE_DATE_TIME,DATE_CREATION,DATE_UPDATE,PLATFORM_NUMBER,CYCLE_NUMBER,DIRECTION,DATA_CENTRE,DATA_MODE,PLATFORM_TYPE,JULD,JULD_QC,JULD_LOCATION,LATITUDE,LONGITUDE,POSITION_QC,PROFILE_PRES_QC,PROFILE_TEMP_QC,PROFILE_PSAL_QC,PRES,PRES_QC,PRES_ADJUSTED,PRES_ADJUSTED_QC,PRES_ADJUSTED_ERROR,TEMP,TEMP_QC,TEMP_ADJUSTED,TEMP_ADJUSTED_QC,TEMP_ADJUSTED_ERROR,PSAL,PSAL_QC,PSAL_ADJUSTED,PSAL_ADJUSTED_QC,PSAL_ADJUSTED_ERROR
0,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,2.8,1,2.76,1,2.4,21.919001,1,21.919001,1,0.002,36.030998,1,36.030998,1,0.019
1,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,4.0,1,3.96,1,2.4,21.931999,1,21.931999,1,0.002,36.030998,1,36.030998,1,0.019
2,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,6.0,1,5.96,1,2.4,21.940001,1,21.940001,1,0.002,36.030998,1,36.030998,1,0.019
3,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,8.0,1,7.96,1,2.4,21.943001,1,21.943001,1,0.002,36.032001,1,36.032001,1,0.019
4,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,10.0,1,9.96,1,2.4,21.938000,1,21.938000,1,0.002,36.030998,1,36.030998,1,0.019


# Clean format of other vars
JULD_QC→Int8, POSITION_QC→Int8

- `JULD_QC` is stored as the string `'1'`; the `isin({1, 2})` filter was comparing integers with strings, so it never matched and silently dropped all records. The `_str_to_int8()` function now converts the values before applying the filter.

In [15]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# PATHS — reuse the names defined in the Config cell, no manual editing needed
INPUT_PARQUET2  = QC_CLEAN_PARQUET
OUTPUT_PARQUET3 = QC_CLEAN2_PARQUET
STR_TO_INT8 = STR_TO_INT8_COLUMNS

def _str_to_int8(series: pd.Series) -> pd.Series:
    """'1' / b'1' / 1 / NaN → Int8 nullable."""
    if series.dtype in ("Int8", "int8"):
        return series
    return (
        series
        .astype(str).str.strip()
        .str.replace(r"^b'(.+)'$", r"\1", regex=True)
        .replace({"nan": pd.NA, "None": pd.NA, "<NA>": pd.NA, "": pd.NA})
        .astype(float).astype("Int8")
    )

pfile  = pq.ParquetFile(INPUT_PARQUET2)
n_rg   = pfile.metadata.num_row_groups
writer = None

for rg in range(n_rg):
    df = pfile.read_row_group(rg).to_pandas()
    for col in STR_TO_INT8:
        if col in df.columns:
            df[col] = _str_to_int8(df[col])
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PARQUET3, table.schema, compression="snappy")
    writer.write_table(table)
    print(f"  Row group {rg + 1}/{n_rg}, good", end="\r")
    del df

if writer:
    writer.close()

print(f"\nSaved in {OUTPUT_PARQUET3}")

  Row group 70/704, good


  Row group 156/704, good


  Row group 380/704, good


  Row group 422/704, good


  Row group 429/704, good


  Row group 704/704, good
Saved in /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_clean2.parquet


In [16]:
import os
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq  

# Reuse the path produced by the previous cell — no manual editing needed
OUTPUT_PARQUET4 = OUTPUT_PARQUET3

# Inicializamos pfile y meta para que Python sepa qué son
pfile = pq.ParquetFile(OUTPUT_PARQUET4)
meta = pfile.metadata

print("-" * 55)
print("OUTPUT PARQUET SUMMARY")
print("-" * 55)
print(f"File          : {OUTPUT_PARQUET4}")
print(f"Size          : {Path(OUTPUT_PARQUET4).stat().st_size / 1024**3:.2f} GB")

print(f"Total rows    : {meta.num_rows:,}")
print(f"Row groups    : {meta.num_row_groups}")
print(f"Columns ({meta.row_group(0).num_columns}):")

#  Show first row group in a dataframe to analyze the values
df_sample = pfile.read_row_group(0).to_pandas()

#  Creates a list to store the analysis of each column 
analisis_columnas = []

for i in range(meta.row_group(0).num_columns):
    col_meta = meta.row_group(0).column(i)
    col_name = col_meta.path_in_schema
    
    # Obtain the kind of data in pandas and unique values 
    tipo_pandas = str(df_sample[col_name].dtype)
    valores_unicos = df_sample[col_name].unique()
    num_unicos = len(valores_unicos)
    
    # Creates a string with a sample of the first 5 posibles values
    muestra_valores = list(valores_unicos[:5])
    if num_unicos > 5:
        muestra_valores.append("...")
        
    analisis_columnas.append({
        "Column": col_name,
        "Type Parquet": str(pfile.schema.column(i).physical_type),
        "Type Pandas": tipo_pandas,
        "Unique Values (Quantity)": num_unicos,
        "Sample possible values ": muestra_valores
    })

#  Convert to dataframe to show as table
df_analisis = pd.DataFrame(analisis_columnas)

with pd.option_context('display.max_columns', None, 'display.max_rows', None, 'display.max_colwidth', None):
    display(df_analisis)

print("\nFirst 5 rows (read from row group 0 only):")
with pd.option_context('display.max_columns', None):
    display(df_sample.head())

print("\n c: Done.")

-------------------------------------------------------
OUTPUT PARQUET SUMMARY
-------------------------------------------------------
File          : /work/drgarcia/Dataset/atlantic_ocean/2021-2025/1.processed/atlantic_2021_2025_clean2.parquet
Size          : 2.21 GB
Total rows    : 689,217,838
Row groups    : 704
Columns (34):


,Column,Type Parquet,Type Pandas,Unique Values (Quantity),Sample possible values
0,DATA_TYPE,BYTE_ARRAY,object,1,[Argo profile]
1,REFERENCE_DATE_TIME,BYTE_ARRAY,object,1,[19500101000000]
2,DATE_CREATION,BYTE_ARRAY,object,4,"[20210101042707, 20210102042709, 20210103022704, 20210104052705]"
3,DATE_UPDATE,BYTE_ARRAY,object,4,"[20260602012447, 20260604064250, 20260527033716, 20260603030153]"
4,PLATFORM_NUMBER,BYTE_ARRAY,object,483,"[5905982, 5905980, 3901219, 6902846, 3901105, ...]"
5,CYCLE_NUMBER,DOUBLE,float64,227,"[81.0, 140.0, 201.0, 200.0, 69.0, ...]"
6,DIRECTION,BYTE_ARRAY,object,2,"[A, D]"
7,DATA_CENTRE,BYTE_ARRAY,object,7,"[AO, IF, BO, JA, ME, ...]"
8,DATA_MODE,BYTE_ARRAY,object,3,"[D, A, R]"
9,PLATFORM_TYPE,BYTE_ARRAY,object,15,"[APEX, S2A, ARVOR, PROVOR_III, SOLO_D_MRV, ...]"



First 5 rows (read from row group 0 only):


,DATA_TYPE,REFERENCE_DATE_TIME,DATE_CREATION,DATE_UPDATE,PLATFORM_NUMBER,CYCLE_NUMBER,DIRECTION,DATA_CENTRE,DATA_MODE,PLATFORM_TYPE,JULD,JULD_QC,JULD_LOCATION,LATITUDE,LONGITUDE,POSITION_QC,PROFILE_PRES_QC,PROFILE_TEMP_QC,PROFILE_PSAL_QC,PRES,PRES_QC,PRES_ADJUSTED,PRES_ADJUSTED_QC,PRES_ADJUSTED_ERROR,TEMP,TEMP_QC,TEMP_ADJUSTED,TEMP_ADJUSTED_QC,TEMP_ADJUSTED_ERROR,PSAL,PSAL_QC,PSAL_ADJUSTED,PSAL_ADJUSTED_QC,PSAL_ADJUSTED_ERROR
0,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,2.8,1,2.76,1,2.4,21.919001,1,21.919001,1,0.002,36.030998,1,36.030998,1,0.019
1,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,4.0,1,3.96,1,2.4,21.931999,1,21.931999,1,0.002,36.030998,1,36.030998,1,0.019
2,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,6.0,1,5.96,1,2.4,21.940001,1,21.940001,1,0.002,36.030998,1,36.030998,1,0.019
3,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,8.0,1,7.96,1,2.4,21.943001,1,21.943001,1,0.002,36.032001,1,36.032001,1,0.019
4,Argo profile,19500101000000,20210101042707,20260602012447,5905982,81.0,A,AO,D,APEX,2021-01-01 23:43:53,1,2021-01-02 00:06:20,-35.038,-47.97,1,A,B,B,10.0,1,9.96,1,2.4,21.938000,1,21.938000,1,0.002,36.030998,1,36.030998,1,0.019



 c: Done.
